# A/B 강약 프롬프트 분석

Weak/Strong 프롬프트 타입별 모델 출력 비교 분석

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
%cd /content/drive/MyDrive/project_05

In [ ]:
!pip install -q unsloth peft transformers accelerate bitsandbytes pandas

In [ ]:
import unsloth   # 반드시 최상단
import json
import torch
import pandas as pd

from unsloth import FastLanguageModel
from peft import PeftModel


df = pd.read_csv("/content/drive/MyDrive/project_05/data/test.csv")

MODEL_CONFIGS = [
    {
        "model_type": "base",
        "model_name": "qwen2.5-7b-instruct-base",
        "base_path": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
        "lora_paths": {
            "pressure_interviewer": None,
            "friendly_interviewer": None,
        },
    },
    {
        "model_type": "lora",
        "model_name": "qwen2.5-7b-instruct-lora",
        "base_path": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
        "lora_paths": {
            "pressure_interviewer": "/content/drive/MyDrive/daon_project/qwen2.5-7B-instruct pressure_lora",
            "friendly_interviewer": "/content/drive/MyDrive/daon_project/qwen2.5-7B-instruct friendly_lora",
        },
    },
]

In [ ]:
def build_eval_prompt(row, prompt_type="strong"):
    inp = json.loads(row["input"]) if isinstance(row["input"], str) else row["input"]

    question = inp["question"]
    candidate_answer = inp["candidate_answer"]
    persona = row["persona"]

    if persona == "pressure_interviewer":
        persona_desc = "압박 면접관"
        persona_rule = """
- 지원자 답변의 모호한 부분을 지적하세요.
- 근거, 수치, 사례, 경험 중 하나를 요구하세요.
- 긍정 표현이나 이모지는 사용하지 마세요.
- 질문은 반드시 1개만 생성하세요.
"""
    else:
        persona_desc = "친절한 면접관"
        persona_rule = """
- 지원자의 답변을 짧게 인정하세요.
- 답변을 구체화하도록 유도하세요.
- 상황, 역할, 행동, 결과, 이유, 사례 중 하나로 확장 질문하세요.
- 질문은 반드시 1개만 생성하세요.
"""

    if prompt_type == "weak":
        global_rule = """
- 자연스럽게 질문하세요.
- 질문은 1개만 생성하세요.
"""
    else:
        global_rule = """
- 반드시 한국어로 작성하세요.
- 영어와 중국어를 사용하지 마세요.
- 질문은 반드시 1개만 생성하세요.
- 지원자 답변에 나온 내용만 근거로 질문하세요.
- persona 스타일을 반드시 유지하세요.
"""

    return f"""당신은 ICT 직무 {persona_desc}입니다.

[GLOBAL RULE]
{global_rule}

[PERSONA RULE]
{persona_rule}

[이전 면접 질문]
{question}

[지원자 답변]
{candidate_answer}

위 답변을 바탕으로 persona에 맞는 꼬리질문 1개만 생성하세요.
꼬리질문 외에는 어떤 설명도 출력하지 마세요."""

In [ ]:
MODEL_CACHE = {}

def load_model(model_cfg, persona):
    if model_cfg["model_type"] == "base":
        cache_key = model_cfg["model_name"]
    else:
        cache_key = f"{model_cfg['model_name']}_{persona}"

    if cache_key in MODEL_CACHE:
        return MODEL_CACHE[cache_key]

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_cfg["base_path"],
        max_seq_length=2048,
        dtype=torch.float16,
        load_in_4bit=True,
    )

    tokenizer.pad_token = tokenizer.eos_token

    if model_cfg["model_type"] == "lora":
        lora_path = model_cfg["lora_paths"][persona]
        model = PeftModel.from_pretrained(model, lora_path)

    FastLanguageModel.for_inference(model)

    MODEL_CACHE[cache_key] = (tokenizer, model)
    return tokenizer, model


def generate_model_output(prompt, model_cfg, persona, max_new_tokens=128):
    tokenizer, model = load_model(model_cfg, persona)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][input_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

In [ ]:
eval_rows = []

PARTIAL_PATH = "/content/drive/MyDrive/project_05/eval_outputs_ab_v2_partial.csv"
FINAL_PATH = "/content/drive/MyDrive/project_05/eval_outputs_ab_v2_final.csv"

for model_cfg in MODEL_CONFIGS:
    for idx, row in df.iterrows():
        inp = json.loads(row["input"]) if isinstance(row["input"], str) else row["input"]

        for prompt_type in ["weak", "strong"]:
            prompt = build_eval_prompt(row, prompt_type=prompt_type)

            model_output = generate_model_output(
                prompt=prompt,
                model_cfg=model_cfg,
                persona=row["persona"],
            )

            eval_rows.append({
                "row_id": row.get("id", idx),
                "row_index": idx,
                "persona": row["persona"],
                "job_role": row.get("job_role", ""),
                "model_type": model_cfg["model_type"],
                "model_name": model_cfg["model_name"],
                "prompt_type": prompt_type,
                "question": inp["question"],
                "candidate_answer": inp["candidate_answer"],
                "reference_output": row.get("output", ""),
                "model_output": model_output,
            })

            if len(eval_rows) % 100 == 0:
                pd.DataFrame(eval_rows).to_csv(
                    PARTIAL_PATH,
                    index=False,
                    encoding="utf-8-sig",
                )
                print("partial saved:", len(eval_rows))

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(
    FINAL_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("saved:", FINAL_PATH)
print(eval_df.groupby(["model_type", "model_name", "prompt_type", "persona"]).size())